**descirption** In this section, feature engineering is performed to prepare data for the recommendation model. Both genre-based and tag-based representations are explored.

for:
one-hot encoding of genres  
feature-matris  
experiment with cosine similarity  
try KNN

# 1. Content-based Movie Recommender - genre

Features:
- Genres converted to text
- TF-IDF vectorization
- Cosine similarity recommendation

In [298]:
#Interactive library
import plotly.express as px

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim

# String matching
from rapidfuzz import process

import re


In [299]:
def normalize_title(title):
    title = title.lower()
    title = title.replace(" ", "")
    title = re.sub(r"\(\d{4}\)", "", title)
    return title

In [300]:
directory_data = "../data" # Adjust the path to your data directory if necessary
movies = pd.read_csv(f"{directory_data}/raw/movies.csv")  # Adjust the path to your data directory if necessary
ratings = pd.read_csv(f"{directory_data}/raw/ratings.csv")
tags = pd.read_csv(f"{directory_data}/raw/tags.csv")

### Content-based recommendation using TF-IDF and cosine similarity

This implementation was inspired by a discussion with Johan Challita on LinkedIn, and is used as a starting point for further exploration.

The initial idea was to begin with a simple content-based similarity recommender as a baseline. In this notebook, a TF-IDF based representation is used together with cosine similarity to identify similar movies.

Rather than implementing a separate KNN model, the nearest neighbours are retrieved directly by ranking movies based on cosine similarity. This approach is equivalent to a nearest neighbour search in the feature space and provides a simple and efficient baseline for recommendation.

### TF-IDF feature representation

To transform the movie data into a numerical representation, TF-IDF vectorization is applied to the genre information.

`TfidfVectorizer` defines how text should be converted into numerical features using the TF-IDF method.

At this stage, the vectorizer is only initialized with chosen settings, such as removing English stop words. It does not yet contain any information about the dataset.

In other words, this step defines *how* the text should be processed, but no transformation has been applied yet.

The genres are first converted into a space-separated text format. This allows them to be treated as tokens and makes it possible to apply standard text processing techniques such as TF-IDF.

---

### TF-IDF transformation (fit_transform)

The `fit_transform()` method performs two steps:

- **fit**: learns the vocabulary from the dataset and calculates how important each word is across all documents  
- **transform**: converts each document into a numerical vector based on these learned values  

The result is a TF-IDF matrix where:
- each row represents a movie  
- each column represents a word (feature)  
- each value represents how important that word is for that movie  

The matrix is stored as a *sparse matrix*, meaning that only non-zero values are stored. This makes it memory efficient, since most words do not appear in most movies.

English stop words are removed during vectorization. While the MovieLens dataset is not limited to English-language movies, this approach is still applicable since genres and many tags are expressed using English terms.

However, this simplification introduces some limitations, as language variations and inconsistencies in user-generated data are not fully addressed.

In [301]:
movies["content"] = movies["genres"].str.replace("|", " ", regex=False)


# Create TF-IDF matrix
tfidf = TfidfVectorizer(stop_words="english")   #for english stop words, you can adjust this for other languages or use a custom list
tfidf_matrix = tfidf.fit_transform(movies["content"])

In [302]:

def recommend_movies_by_genres(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print(f"Movie could not be found")
        return None
    if len(matches) > 1:
        print("multiple movies matched your search: ")
        print(matches[["title"]].head(10))

    
    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(tfidf_matrix[movie_index], tfidf_matrix).flatten()
       

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres"]].head(n).copy()

    print(f"Recommendations for '{title}':")
    print(f"Movies with similarity > 0: {(similarity_score > 0).sum() - 1}")
    print(f"Movies with similarity > 0.1: {(similarity_score > 0.1).sum() - 1}")
    print(f"Movies with similarity > 0.3: {(similarity_score > 0.3).sum() - 1}")
    print(f"Movies with similarity > 0.5: {(similarity_score > 0.5).sum() - 1}")


    return result

In [303]:
recommend_movies_by_genres("kung fu hustle", 5)

Recommendations for 'Kung Fu Hustle (Gong fu) (2004)':
Movies with similarity > 0: 30660
Movies with similarity > 0.1: 30660
Movies with similarity > 0.3: 25256
Movies with similarity > 0.5: 12866


,title,genres
64827,Take Home Pay (2019),Action|Comedy
70281,Cats & Dogs 3: Paws Unite (2020),Action|Comedy
4479,Disorganized Crime (1989),Action|Comedy
58972,Once Upon a Deadpool (2018),Action|Comedy
55743,Surge of Power: Revenge of the Sequel (2018),Action|Comedy


#Note to me: explore till KNN-solution! What differnece will it make?

The current model is based on TF-IDF representations and cosine similarity, which rely on word overlap rather than true semantic understanding. As a result, many movies receive relatively high similarity scores because they share common features (genres). This leads to a broad similarity structure where many movies are considered similar, even if they are not closely related in meaning.

# 2. Content-based Movie Recommender - genre and tags

Features:
- Genres converted to text
- TF-IDF vectorization
- combination tags and genre
- Cosine similarity recommendation

**tags per movie**

In [304]:
tags_per_movie = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index()
    .rename(columns={"tag": "tags_combined"})
)

In [305]:
print(movies.columns)

print(tags_per_movie["tags_combined"].isna().sum())

Index(['movieId', 'title', 'genres', 'content'], dtype='str')
0


**Merge tags into movies**

In [306]:
movies = movies.merge(tags_per_movie, on="movieId", how="left")



**fill missing tags**

In [307]:
movies["tags_combined"] = movies["tags_combined"].fillna("")

**Clean genres**

In [308]:
movies["genres_clean"] = movies["genres"].apply(
    lambda x: "" if x == "(no genres listed)" else x
)

**clean title**

In [309]:
movies["clean_title"] = movies["title"].apply(normalize_title)



**Build cobined test column**

*obs! Kolla genre_clean om det verkligen finns "-" eller "" i datan!*

**Create new content**  

In this step, a new column is created by combining cleaned genres and tags into a single text feature.  
This column is used as input for the TF-IDF vectorizer.  

The original columns are kept unchanged to preserve the raw data and allow traceability.  

This step constructs the feature used by the content-based model.

In [310]:
movies["genres_clean"] = (
    movies["genres"]
    .str.replace("(no genres listed)", "", regex=False)
    .str.replace("|", " ", regex=False)
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["tags_clean"] = (
    movies["tags_combined"]
    .fillna("")
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["content_with_tags"] = (
    movies["genres_clean"]
    + " "
    + movies["tags_clean"].str.replace(r"\b\d+\b", 
    "", 
    regex=True
    )
)

movies["content_with_tags"] = (
    movies["content_with_tags"]
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)   # Only keep latin letters and spaces
    .str.replace(r"\s+", " ", regex=True)          # Replace multiple spaces with a single space
    .str.strip()                                   # Remove leading and trailing spaces
)

**Handle different alphabets, signs and numbers**

A limitation of this approach is that it removes potentially meaningful non-English tags. However, given the scope of the model, this trade-off was made to improve consistency and interpretability.

Since the model uses English stop words and does not include language detection, non-Latin characters were removed from the dataset. This reduces noise from multilingual user-generated tags and ensures a more consistent feature space for similarity calculations.

In [322]:
#  TF-IDF vectorization on combined genres + tags
tfidf_tags = TfidfVectorizer(stop_words="english")
tfidf_matrix_tags = tfidf_tags.fit_transform(movies["content_with_tags"])
feature_names = tfidf_tags.get_feature_names_out()

**Function för genre + tags**

In [312]:
def recommend_movies_by_genres_and_tags(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print("Movie could not be found")
        return None
    
    if len(matches) > 1:
        print("Multiple movies found. Please be more specific: ")
        print(matches[["title"]].head(10))

    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(
        tfidf_matrix_tags[movie_index],
        tfidf_matrix_tags
    ).flatten()

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres", "tags_combined"]].head(n).copy()

    return result

    

**test function**

In [313]:
recommend_movies_by_genres_and_tags("Toy Story", 5)


Multiple movies found. Please be more specific: 
                                           title
0                               Toy Story (1995)
3021                          Toy Story 2 (1999)
14815                         Toy Story 3 (2010)
20507                 Toy Story of Terror (2013)
22654  Toy Story Toons: Hawaiian Vacation (2011)
22655          Toy Story Toons: Small Fry (2011)
24101    Toy Story Toons: Partysaurus Rex (2012)
24103          Toy Story That Time Forgot (2014)
60793                         Toy Story 4 (2019)


,title,genres,tags_combined
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,whimsical cgi Disney Pixar very good but the f...
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,animation Disney Pixar acting circus coming of...
14815,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Pixar Pixar Annemari dolls escape good versus ...
39883,Finding Dory (2016),Adventure|Animation|Comedy,adventure amnesia Animation boring dumb famil...
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,cute funny innovative Disney Pixar cute funny ...


**check which words dictate the recommendation?**

**Choose a movie**

In [314]:
movie_title = "Toy Story"
matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]


In [315]:
print(matches[["title"]].head())

print(movies[movies["title"] == movie_title])

                                           title
0                               Toy Story (1995)
3021                          Toy Story 2 (1999)
14815                         Toy Story 3 (2010)
20507                 Toy Story of Terror (2013)
22654  Toy Story Toons: Hawaiian Vacation (2011)
Empty DataFrame
Columns: [movieId, title, genres, content, tags_combined, genres_clean, clean_title, tags_clean, content_with_tags]
Index: []


In [316]:
print(movies[movies["title"].str.contains("Toy", case=False, na=False)][["title"]].head())


                                      title
0                          Toy Story (1995)
1928                Babes in Toyland (1961)
2162                            Toys (1992)
2389  Dry Cleaning (Nettoyage à sec) (1997)
2993                Babes in Toyland (1934)


**Find index**

In [317]:
movie_index = matches.index[0]

print(movie_index)

0


**rating-foundation**  
-movieID  
-average_rating  
-rating_count  
  
later:  
weighted_count


grupby("movieId") - collects all ratings for the same movie  
  
mean - calculates the mean rating for the movie.  
  
count - counts how many rating the movie has got.  


In [318]:
user_counts = ratings.groupby("userId").size().reset_index(name="user_rating_count")

#merge in ratings
ratings = ratings.merge(user_counts, on="userId", how="left")

#create userweight
ratings["user_weight"] = 1 / np.log1p(ratings["user_rating_count"])

#create weighted rating per row
ratings["weighted_user_rating"] = ratings["rating"] * ratings["user_weight"]



*ratings_summary*  

In [319]:
ratings_summary = (
    ratings.groupby("movieId")
    .agg(
        weighted_sum = ("weighted_user_rating", "sum"),
        weight_total = ("user_weight", "sum"),
        rating_count = ("rating", "count")
    )
    .reset_index()
)

ratings_summary["average_rating"] = (
    ratings_summary["weighted_sum"] / ratings_summary["weight_total"]
)

**Merge with movies**  


In [320]:
movies = movies.drop(
    columns = ["average_rating", "rating_count", "weighted_rating"],
    errors = "ignore"
)

movies = movies.merge(
    ratings_summary[["movieId", "average_rating", "rating_count" ]],
    on= "movieId",
    how = "left"
)

movies["average_rating"] = movies["average_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0)

print(movies[["title", "average_rating", "rating_count"]].head())

                                title  average_rating  rating_count
0                    Toy Story (1995)        3.897434       76813.0
1                      Jumanji (1995)        3.313997       30209.0
2             Grumpier Old Men (1995)        3.220433       15820.0
3            Waiting to Exhale (1995)        2.925581        3028.0
4  Father of the Bride Part II (1995)        3.121359       15801.0


In [321]:
print(movies[["title", "average_rating", "rating_count", "weighted_rating"]].head())

KeyError: "['weighted_rating'] not in index"

**weighted rating**

weighted_rating = v/(v+m)R+m/(v+m)C  

R = average_rating  
V = rating_count  
c = mean_rating in the whole data  
m = a threshold for how many ratings are required before we can trust the movie’s own average more (how many is "a lot" of ratings)

count C and M

In [ ]:
c = movies["average_rating"].mean()

m = movies["rating_count"].quantile(0.90)

print("c:", c)
print("m:", m)

c: 2.897889261088383
m: 263.0


Create column *weighted_rating*

In [ ]:
movies["weighted_rating"] = (
    (movies["rating_count"] / (movies["rating_count"] + m)) * movies["average_rating"]
    + (m / (movies["rating_count"] + m)) * c
)



**ranking adjustment: popularity and user bias**
1%of users has given 13% of all ratings.  
The average user only rates movies they seam to enjot.  
Wellknown and popular movies has far more ratings.  
I want to adjust these biases. 

In [ ]:
movies

,movieId,title,genres,content,tags_combined,genres_clean,clean_title,tags_clean,content_with_tags,average_rating,rating_count,ratings_count,weighted_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,animation friendship toys animation Disney Pix...,adventure animation children comedy fantasy,toystory,animation friendship toys animation disney pix...,adventure animation children comedy fantasy an...,3.893508,76813.0,76813.0,3.890110
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,animals based on a book fantasy magic board ga...,adventure children fantasy,jumanji,animals based on a book fantasy magic board ga...,adventure children fantasy animals based on a ...,3.278179,30209.0,30209.0,3.274896
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,sequel moldy old old age old men wedding old p...,comedy romance,grumpieroldmen,sequel moldy old old age old men wedding old p...,comedy romance sequel moldy old old age old me...,3.171271,15820.0,15820.0,3.166800
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,characters chick flick girl movie characters c...,comedy drama romance,waitingtoexhale,characters chick flick girl movie characters c...,comedy drama romance characters chick flick gi...,2.868395,3028.0,3028.0,2.870752
4,5,Father of the Bride Part II (1995),Comedy,Comedy,family pregnancy wedding 4th wall aging baby d...,comedy,fatherofthebridepartii,family pregnancy wedding 4th wall aging baby d...,comedy family pregnancy wedding th wall aging ...,3.076957,15801.0,15801.0,3.074025
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86532,288967,State of Siege: Temple Attack (2021),Action|Drama,Action Drama,,action drama,stateofsiege:templeattack,,action drama,3.500000,1.0,1.0,2.900170
86533,288971,Ouija Japan (2021),Action|Horror,Action Horror,,action horror,ouijajapan,,action horror,0.500000,1.0,1.0,2.888806
86534,288975,The Men Who Made the Movies: Howard Hawks (1973),Documentary,Documentary,,documentary,themenwhomadethemovies:howardhawks,,documentary,4.000000,1.0,1.0,2.902064
86535,288977,Skinford: Death Sentence (2023),Crime|Thriller,Crime Thriller,,crime thriller,skinford:deathsentence,,crime thriller,3.000000,1.0,1.0,2.898276


*test*

In [ ]:
print(movies[["title", "average_rating", "rating_count", "weighted_rating"]].head())

                                title  average_rating  rating_count  \
0                    Toy Story (1995)        3.893508       76813.0   
1                      Jumanji (1995)        3.278179       30209.0   
2             Grumpier Old Men (1995)        3.171271       15820.0   
3            Waiting to Exhale (1995)        2.868395        3028.0   
4  Father of the Bride Part II (1995)        3.076957       15801.0   

   weighted_rating  
0         3.890110  
1         3.274896  
2         3.166800  
3         2.870752  
4         3.074025  


*test top movies*

In [ ]:
movies[["title", "average_rating", "rating_count", "weighted_rating"]].sort_values(
    by="weighted_rating", ascending=False
).head(10)

,title,average_rating,rating_count,weighted_rating
314,"Shawshank Redemption, The (1994)",4.416792,122296.0,4.413533
41017,Planet Earth (2006),4.448093,3015.0,4.323717
840,"Godfather, The (1972)",4.326603,75004.0,4.321610
61176,Parasite (2019),4.329946,12399.0,4.300201
46156,Band of Brothers (2001),4.423986,2835.0,4.294430
46305,Planet Earth II (2016),4.451739,2041.0,4.274368
49,"Usual Suspects, The (1995)",4.267865,72893.0,4.262940
1190,"Godfather: Part II, The (1974)",4.269510,47271.0,4.261921
1173,12 Angry Men (1957),4.267158,22730.0,4.251496
522,Schindler's List (1993),4.242337,84232.0,4.238152
